<div align="right"><a href="S35-SpectralEstimators.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

## Übung: Vergleich der Schätzalgorithmen

In dieser Übung sollen drei Algorithmen zur Schätzung der Koeffizienten eines AR($N$)-Modells (Autokorrelationsmethode, Autokovarianzmethode und Burg-Algorithmus) hinsichtlich ihres **Bias** und ihrer **Varianz** verglichen werden.

Zusätzlich wird ein Verfahren untersucht, das auf der Autokorrelationsmethode basiert und zur Auswertung der Daten ein **Hamming-Fenster** anstelle eines rechteckigen Fensters verwendet
(MATLAB-Funktion: `hamming(·)`).

**Gegeben:** (All-Pol-)Übertragungssystem

$$
H(z) = \frac{1}{A(z)}
$$

mit

$$
\begin{aligned}
A(z) =;& 1 - 2.2346 z^{-1} + 2.3921 z^{-2} - 2.0324 z^{-3} \
&+ 1.2865 z^{-4} - 0.4954 z^{-5} + 0.2322 z^{-6} \
&- 0.2554 z^{-7} - 0.1848 z^{-8}
\end{aligned}
$$

der Ordnung $m = 8$, das durch einen stationären, mittelwertfreien, weißen Prozess
$N(k)$ mit der Leistung

$$
\sigma_{\pi N}^2 = 1
$$

angeregt wird.

<div style="text-align:center">
  <img src="./../pngs/l15sys.eps" width="70%">
</div>

---

### Aufgaben

**a)**
Berechnen Sie zunächst das Pol-Nullstellen-Diagramm des Systems mit dem Befehl
`zplane(·)`.

Für eine genauere Analyse des Systems bestimmen Sie die exakten PARCOR-Koeffizienten

$$
\gamma_i, \quad i = 1 \ldots 8 .
$$

Bestimmen Sie dazu die **unverzerrte Autokorrelationsfolge** $r_{\pi XX}(\kappa)$
mithilfe der Impulsantwort des Systems und lösen Sie die exakte
Yule–Walker-Gleichung (9.3.17) unter Anwendung der **Levinson–Durbin-Rekursion**.

---

**b)**
Schätzen Sie das Leistungsdichtespektrum des Prozesses $X(k)$ mithilfe eines
AR($8$)-Modells unter Verwendung der oben beschriebenen Methoden.

Verwenden Sie hierzu **1000 Realisierungen** des stochastischen Prozesses $X(k)$,
wobei jedes Stichprobenintervall die Länge

$$
N = 64
$$

Samples besitzen soll.

Um das Einschwingverhalten des Systems $H(z)$ zu berücksichtigen, verwenden Sie mindestens

$$
2^{11} + 64
$$

Samples des Prozesses $N(k)$.
Nutzen Sie den Befehl `filter(·)`, um das weiße Rauschen zu filtern.

Stellen Sie die folgenden Ergebnisse grafisch dar bzw. geben Sie die berechneten Werte an:

* **AR-Koeffizienten**:
  exakter Wert $a_i$ / Mittelwert $\bar{a}_i$ / mittlere quadratische Abweichung

  $$
  \frac{1}{n} \sum_{i=0}^{n} \lvert \bar{a}_i - a_i \rvert^2
  $$

  Standardabweichung $\sigma$ sowie die $2\sigma$-Grenzen

* **Leistungsdichtespektrum (PDS)**:
  exaktes und gemitteltes Leistungsdichtespektrum sowie die $2\sigma$-Grenzen

Vergleichen Sie die verschiedenen Algorithmen hinsichtlich **Bias** und **Varianz** der Schätzung.

Die folgenden MATLAB-Funktionen realisieren die unterschiedlichen Algorithmen:

* Autokorrelationsmethode: `lyw(·)`
* Autokovarianzmethode: `lcov(·)`
* Burg-Algorithmus: `lburg(·)`

In [ ]:
# BEGIN SOLUTION
MA = [1];
AR = [1.0000   -2.2346    2.3921   -2.0324,...
      1.2865   -0.4954    0.2322   -0.2554    0.1848];
NFFT = 2^6;
n = length(AR)-1;

N = [64];
N_vor = 2^11;
mc_run = 2e3;

% #####  Teilaufgabe a:  Einschwingverhalten #####
% exakte AKF (eingeschw. Zustand)

% # grafische Ausgabe # %
figure;
[h1,h2,h3] = zplane(MA,AR); 
set(h1,'Visible','off');
title(sprintf('Pole des AR(%d)-Modells',length(AR)-1)); 
grid;
xlabel('Realteil'); 
ylabel('Imaginaerteil');

h_arma = impz(MA,AR);
rxx_all = xcorr(h_arma);
index = (length(rxx_all)+1)/2;
rxx = rxx_all(index:index+n);
var = rxx(1);
g(1) = rxx(2)/rxx(1);
var = [var (1-abs(g(1)).^2)*var];
ARcoeff = [1 -g(1)];

for i = 2:n
    g(i) = (ARcoeff*rxx(i+1:-1:2))./var(i);
    var = [var (1-abs(g(i)).^2)*var(i)];
    ARcoeff = [ARcoeff 0]-fliplr(conj([ARcoeff 0]))*g(i);
end;

disp('Exakten PARCOR-Koeffizienten:');
disp(sprintf('%1.4f\n',g));
input('....Zum Fortfahren: RETURN druecken');

% #####  Teilaufgabe b:  Exkates LDS #####
Omega_norm  = 0:1/NFFT:1-1/NFFT;
Sxx_ex = abs(fft(MA,NFFT)./fft(AR,NFFT)).^2;
Sxx_ar = zeros(mc_run,NFFT);
Coeff = zeros(mc_run,length(AR));

disp('.......Bitte warten: Berechnung erfolgt.....');

for run = 1:4
    for run1 = 1:mc_run
        noise = randn(1,20*N+N_vor);
        x_all = filter(MA,AR,noise);

        switch run
            case 1; 
                x = x_all(N_vor+1:N_vor+1+N);
                [ARcoeff] = lyw(x,n,1); 
                str = 'YW';
            case 2; 
                x = x_all(N_vor+1:N_vor+1+N);
                x_ham = x.*hamming(length(x)).';
                [ARcoeff] = lyw(x_ham,n,1); 
                str = 'YW-Ham.';
            case 3; 
                x = x_all(N_vor+1:N_vor+1+N);
                [ARcoeff] = lcov(x,n); 
                str = 'Kovar.';
            case 4; 
                x = x_all(N_vor+1:N_vor+1+N);
                [ARcoeff] = lburg_algo(x,N,n,MA,AR); 
                str = 'Burg';
        end;
        
        Coeff(run1,:) = ARcoeff;
        Sxx_ar(run1,:) = abs(ones(1,NFFT)./fft(ARcoeff,NFFT)).^2;
    end;

    Coeff_mean = mean(Coeff,1);
    Coeff_std = std(Coeff,1);
    C2sig_o = Coeff_mean+2*Coeff_std;
    C2sig_u = Coeff_mean-2*Coeff_std;
    Coeff_lst = mean(abs(AR-Coeff_mean).^2);
    Sxx_mean = mean(Sxx_ar,1);
    Sxx_2sigma = Sxx_mean+2*std(Sxx_ar,1);
    
    figure;
    h1=stem([0:n],AR,'ro');  hold on;
    h2=stem([0:n],Coeff_mean,'bx');  hold on;
    h3=stem([0:n],C2sig_o,'go','filled'); hold on;
    h4=stem([0:n],C2sig_u,'go','filled');
    set(h1,'Linestyle','none');
    set(h2,'Linestyle','none');
    set(h3,'Linestyle','none');
    set(h4,'Linestyle','none')
    x_lines=[[0:n]; [0:n]; nan*ones(1,n+1)];
    stddev_lines=[C2sig_u; C2sig_o; nan*ones(1,n+1)];
    plot(x_lines(:),stddev_lines(:),'g');
    title([sprintf('%s-AR(%d)-Koeffizienten; N=%d',str,n,N)]);
    xlabel('i'); ylabel('Koeffizienten'); grid; axis([0 length(AR)-1 -4 4]);
    legend([h1(1),h2(1),h3(1)],'Exakt',...
        ['Mittel./E\{e^2\}=',sprintf('%1.2e',Coeff_lst)],...
        ['2\sigma-Gr./ \sigma=',sprintf('%1.2f',mean(Coeff_std))]);
    hold off;

    % # grafische Ausgabe # %
    figure;
    h1 = plot(Omega_norm, 10*log10(Sxx_ex/max(Sxx_ex)),'r--');     hold on;
    h2 = plot(Omega_norm, 10*log10(Sxx_mean/max(Sxx_ex)),'b-');    hold on;
    h3 = plot(Omega_norm, 10*log10(Sxx_2sigma/max(Sxx_ex)),'g-.'); hold off;
    ylabel('S_{xx}(e^{j\Omega}) in dB');
    legend([h1,h2,h3],'Exakt','Mittel.','2\sigma-Gr.');
    title(sprintf('LDS von %s-AR(%d); N=%d',str,n,N));  
    xlabel('\Omega/2\pi');
    grid;
end;
# END SOLUTION

## Lösung: Vergleich der Schätzalgorithmen

### a)

Die Abbildung zeigt das Pol-Nullstellen-Diagramm des Systems $H(z)$. Es handelt sich um ein reellwertiges System, bei dem sich ein Polpaar bei

$$
\Omega / 2\pi \approx 0.05
$$

sehr nahe am Einheitskreis befindet.

Für den zugrunde liegenden AR(8)-Prozess ergeben sich bei Verwendung der exakten Yule–Walker-Gleichung die folgenden PARCOR-Koeffizienten:

$$
\gamma_1 = 0.9785,;
\gamma_2 = -0.9279,;
\gamma_3 = 0.1531,;
\gamma_4 = -0.6974,
$$

$$
\gamma_5 = 0.1815,;
\gamma_6 = -0.1563,;
\gamma_7 = -0.1631,;
\gamma_8 = -0.1848 .
$$

Es ist zu erkennen, dass insbesondere die ersten beiden PARCOR-Koeffizienten nahezu den Wert Eins besitzen und damit das Verhalten des AR-Prozesses dominieren. Dieser AR-Prozess entspricht einem typischen Sprachsignalmodell.

<div style="text-align:center">
  <img src="./../pngs/ldsv9a04_a1.png">
</div>

---

### b)

Im Folgenden werden die wesentlichen Eigenschaften der bekannten spektralen Schätzverfahren zusammengefasst. Es wird ein Stichprobenintervall der Länge $N$ des Prozesses $X(k)$ angenommen. Für die Diskussion wird die **Leistung des Prädiktionsfehlers** betrachtet.

Aufgrund der inversen Beziehung zwischen einem AR($n$)-Modell und einem Prädiktor ist die Minimierung der Prädiktionsfehlerleistung äquivalent zur Bestimmung der Parameter eines AR($n$)-Prozesses. Mit anderen Worten wird dem Prozess $X(k)$ ein Prädiktor zugeordnet, dessen Koeffizienten so gewählt werden, dass die Prädiktionsfehlerleistung minimal ist bzw. der Leistung des zugrunde liegenden weißen Rauschprozesses $N(k)$ entspricht.

---

#### Yule–Walker- / Autokorrelationsmethode

* Der Prozess $X(k)$ wird außerhalb des Stichprobenintervalls als Null angenommen (rechteckige Fensterung)
* Die Einschwingzeit des Prädiktors wird in die Minimierung einbezogen
* Es wird die **verzerrte** Autokorrelationsfolge $\hat{r}_{\pi XX}(\kappa)$ verwendet
* Die Lösung basiert auf einer hermiteschen Toeplitz-Matrix
* Die Schätzung ist **verzerrt**, das AR-Modell ist jedoch stets **minimalphasig**

---

#### Autokovarianzmethode

* Der Prozess $X(k)$ wird außerhalb des Stichprobenintervalls als nicht definiert betrachtet
* Die Einschwingzeit des Prädiktors wird **nicht** in die Minimierung einbezogen
* Es wird die **unverzerrte** Autokorrelationsfolge $r_{\pi XX}(\kappa)$ verwendet
* Es existiert keine Toeplitz-Matrix
* Die Schätzung ist **unverzerrt**, das AR-Modell ist jedoch nicht notwendigerweise **minimalphasig**

---

#### Burg-Algorithmus

* Gemeinsame Minimierung der Vorwärts- und Rückwärts-Prädiktionsfehlerleistung
* Der Prozess $X(k)$ wird außerhalb des Stichprobenintervalls als nicht definiert betrachtet
* Die Einschwingzeit des Prädiktors wird in die Minimierung einbezogen
* Die Schätzung ist **unverzerrt** und das AR-Modell ist stets **minimalphasig**

---

#### Autokorrelationsmethode mit Hamming-Fenster

Anstelle eines rechteckigen Fensters wird auf den Prozess $X(k)$ ein Hamming-Fenster der Länge $N$ angewendet.

Anstatt die Korrelation zweier rechteckiger Fenster—die zur Bartlett-Fensterung führt—zur Berechnung der verzerrten Autokorrelationsfolge zu verwenden, wird die Korrelation zweier Hamming-Fenster eingesetzt. Dadurch entsteht für kleine Verzögerungen $\kappa$ ein geringer Bias der Autokorrelationswerte $\hat{r}_{\pi XX}(\kappa)$, während größere $\kappa$-Werte stärker gewichtet werden.

Zur Veranschaulichung dieses Verhaltens werden die unverzerrten Korrelationsfunktionen beider Fenster dargestellt. Dieses Verfahren wird häufig in der Sprachsignalverarbeitung eingesetzt, da Sprachsignale typischerweise einen dominanten ersten PARCOR-Koeffizienten besitzen, der im Wesentlichen durch $\hat{r}*{\pi XX}(0)$ und $\hat{r}*{\pi XX}(1)$ bestimmt wird.

<div style="text-align:center">
  <img src="./../pngs/ldsv9a04fenster.png">
</div>

---

Die Abbildungen auf der linken Seite zeigen die Ergebnisse der Koeffizientenschätzung, während auf der rechten Seite die zugehörigen Leistungsdichtespektren dargestellt sind.

<div style="text-align:center">
  <img src="./../pngs/ldsv9a04_b11.png">
  <img src="./../pngs/ldsv9a04_b12.png">
</div>

Die Ergebnisse der Autokorrelationsmethode („YW-AR(8)“) bestätigen das Vorhandensein eines Bias bei dieser Schätzung. So liegen einige exakte Koeffizienten, beispielsweise $a_3$, außerhalb der $2\sigma$-Grenzen. Dieses Verhalten spiegelt sich auch in der relativ hohen Prädiktionsfehlerleistung

$$
E{e^2} = 1.18
$$

wider. Das Leistungsdichtespektrum wird nur unzureichend approximiert, und der kritische Pol bei $\Omega / 2\pi \approx 0.05$ wird fehlerhaft modelliert.

---

<div style="text-align:center">
  <img src="./../pngs/ldsv9a04_b21.png">
  <img src="./../pngs/ldsv9a04_b22.png">
</div>

Bei Verwendung der mit einem Hamming-Fenster modifizierten Autokorrelationsmethode („YW-HAM.-AR(8)“) zeigt sich für Prozesse mit großen PARCOR-Koeffizienten $\gamma_1$ und $\gamma_2$ eine deutliche Reduktion des Bias.

So liegt der Koeffizient $a_3$, der zuvor außerhalb der $2\sigma$-Grenzen lag, nun innerhalb dieser Grenzen. Die mittlere Prädiktionsfehlerleistung sinkt auf

$$
E{e^2} = 0.25 .
$$

Ein Nachteil der Hamming-Gewichtung ist die erhöhte Standardabweichung $\sigma$. Das geschätzte Leistungsdichtespektrum zeigt jedoch eine deutliche Verbesserung.

---

<div style="text-align:center">
  <img src="./../pngs/ldsv9a04_b31.png">
  <img src="./../pngs/ldsv9a04_b32.png">
</div>

Eine unverzerrte Schätzung der Koeffizienten wird mit der Autokovarianzmethode („cov.-AR(8)“) erreicht. Die geringen Abweichungen zwischen exakten und gemittelten Werten sind auf die endliche Anzahl von Realisierungen ($1000$) zurückzuführen. Die Standardabweichung verhält sich ähnlich wie bei der modifizierten Autokorrelationsmethode.

Damit stellt die Autokovarianzmethode eine interessante Alternative zur spektralen Schätzung dar, was auch durch die Approximation des Leistungsdichtespektrums bestätigt wird. Ihr Nachteil besteht in der fehlenden Garantie eines minimalphasigen Prädiktionsfilters.

---

<div style="text-align:center">
  <img src="./../pngs/ldsv9a04_b41.png">
  <img src="./../pngs/ldsv9a04_b42.png">
</div>

Die obigen Abbildungen zeigen die Ergebnisse des Burg-Algorithmus („Burg-AR(8)“). Die gemeinsame Minimierung der Vorwärts- und Rückwärts-Prädiktionsfehlerleistung erweist sich als besonders geeignet. Der fehlende Bias ist deutlich erkennbar, und das Leistungsdichtespektrum wird nahezu ebenso gut approximiert wie mit der Autokovarianzmethode.

Hinsichtlich der Schätzvarianz verhält sich der Burg-Algorithmus im betrachteten Beispiel ähnlich wie die Autokovarianz- bzw. die modifizierte Autokorrelationsmethode.
